# 🔍 What is RAG? (Retrieval-Augmented Generation)

Welcome to the RAG module! This is your starting point for understanding one of the most practical and powerful techniques in modern AI. Whether you're a developer, a curious student, or just someone who wants to know how AI chatbots can answer questions about *your* documents — you're in the right place.

## 🎯 What You’ll Learn

By the end of this notebook, you’ll understand:
- What RAG is and why we need it
- How computers understand text (embeddings)
- How to measure if two texts are similar (vector similarity)
- The complete RAG pipeline from start to finish

**Prerequisites:** Just curiosity! We’ll explain everything from scratch.

---
## 📚 The Problem: Why LLMs Need Help

### Analogy: The Closed-Book vs Open-Book Test

Imagine you're a student about to take a big exam. There are two scenarios:

**Scenario 1: Closed-Book Test 📕**
- You can only use what you memorized *before* the test
- If a question is about something you never studied, you have to guess
- You might *feel* confident about a wrong answer (you misremember a fact)
- You can't look up anything that happened after you stopped studying

**Scenario 2: Open-Book Test 📖**
- You have your notes, textbooks, and references right in front of you
- You can look up specific facts to make sure you're right
- You can reference the *latest* information
- Your answers are grounded in actual sources

### LLMs Are Closed-Book Students!

Large Language Models (like GPT-4, Claude, etc.) are incredibly smart, but they have real limitations:

| Problem | Explanation |
|---------|-------------|
| 📅 **Knowledge Cutoff** | They were trained on data up to a certain date. They don't know about anything that happened after that. |
| 👻 **Hallucinations** | They can make up facts that sound completely convincing but are totally wrong. |
| 🔒 **No Access to YOUR Data** | They haven't read your company's internal documents, your personal notes, or your private database. |
| 📦 **Static Knowledge** | Once trained, their knowledge is frozen. Updating it requires expensive retraining. |

### RAG = Giving the LLM an Open Book!

**RAG (Retrieval-Augmented Generation)** solves these problems by giving the LLM relevant reference material *at the time it answers your question*. Instead of relying solely on memorized knowledge, the LLM can look things up!

> 💡 **Think of it this way:** RAG doesn't make the LLM smarter — it gives the LLM better reference material to work with.

---
## 🧩 What is RAG? (Simple Explanation)

Let’s break down the name **R-A-G**:

| Letter | Word | Simple Meaning |
|--------|------|----------------|
| **R** | Retrieval | **Find** relevant information from a collection of documents |
| **A** | Augmented | **Enhance** the LLM’s input by adding that information |
| **G** | Generation | **Generate** an answer using both the question and the retrieved info |

### 🏢 Analogy: RAG is Like a Research Assistant

Imagine you hire a brilliant research assistant. Here’s how they work:

1. 🙋 **You ask a question:** "What were our Q3 sales in Europe?"
2. 🔎 **They search through your files:** They go to the filing cabinet and look for relevant reports
3. 📄 **They find the most relevant pages:** They pull out the Q3 report and the Europe regional summary
4. ✍️ **They write an answer:** They read those documents and write you a clear answer based on what they found

That’s exactly what RAG does! The LLM is the research assistant, and the document collection is the filing cabinet.

### Without RAG vs With RAG

**Without RAG:**
```
You: "What is our company's vacation policy?"
LLM: "Most companies offer 2-3 weeks of PTO..." (generic guess)
```

**With RAG:**
```
You: "What is our company's vacation policy?"
RAG System: [Retrieves your company's HR handbook]
LLM: "According to your employee handbook, full-time employees 
       receive 20 days of PTO per year, plus 10 holidays..." (specific, accurate answer)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ============================================================
# RAG Pipeline Overview Diagram
# Shows the high-level flow: Question -> Retrieval -> LLM -> Answer
# ============================================================

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')

# Define box properties for each stage
boxes = [
    {'xy': (0.5, 1.2), 'width': 2.2, 'height': 1.6, 'color': '#4FC3F7',
     'label': 'User\nQuestion', 'emoji': '\U0001f64b'},
    {'xy': (3.5, 1.2), 'width': 2.2, 'height': 1.6, 'color': '#FFB74D',
     'label': 'Retrieval\nSystem', 'emoji': '\U0001f50e'},
    {'xy': (6.5, 1.2), 'width': 2.2, 'height': 1.6, 'color': '#81C784',
     'label': 'Relevant\nDocuments', 'emoji': '\U0001f4c4'},
    {'xy': (9.5, 1.2), 'width': 2.2, 'height': 1.6, 'color': '#CE93D8',
     'label': 'LLM', 'emoji': '\U0001f9e0'},
    {'xy': (12.5, 1.2), 'width': 1.2, 'height': 1.6, 'color': '#EF5350',
     'label': 'Answer', 'emoji': '\u2705'},
]

for box in boxes:
    # Draw rounded rectangle
    rect = mpatches.FancyBboxPatch(
        box['xy'], box['width'], box['height'],
        boxstyle='round,pad=0.1', facecolor=box['color'],
        edgecolor='gray', linewidth=2, alpha=0.85
    )
    ax.add_patch(rect)
    # Add label text centered in box
    cx = box['xy'][0] + box['width'] / 2
    cy = box['xy'][1] + box['height'] / 2
    ax.text(cx, cy - 0.15, box['label'], ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
    ax.text(cx, cy + 0.55, box['emoji'], ha='center', va='center', fontsize=18)

# Draw arrows between boxes
arrow_style = dict(arrowstyle='->', lw=2.5, color='#333333')
arrow_xs = [(2.7, 3.5), (5.7, 6.5), (8.7, 9.5), (11.7, 12.5)]
for (x_start, x_end) in arrow_xs:
    ax.annotate('', xy=(x_end, 2.0), xytext=(x_start, 2.0),
                arrowprops=arrow_style)

plt.title('\U0001f50d RAG Pipeline Overview', fontsize=18, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\n\U0001f4ca How RAG works at a glance:")
print("1. You ask a question")
print("2. The retrieval system searches for relevant documents")
print("3. The most relevant documents are selected")
print("4. The LLM reads your question + the documents")
print("5. It generates an answer grounded in those documents")

---
## 📍 How Computers Understand Text: Embeddings

Before we can build a RAG system, we need to answer a fundamental question: **How does a computer know that two pieces of text are similar?**

Computers don't understand words the way we do. They need numbers. So we need a way to convert text into numbers that *capture meaning*.

### 🌍 Analogy: GPS Coordinates for Words

Think about how GPS works:
- Every place on Earth has **coordinates** (latitude, longitude)
- Places that are **close together** have **similar coordinates**
  - New York (40.7, -74.0) and New Jersey (40.7, -74.2) are close
- Places that are **far apart** have **very different coordinates**
  - New York (40.7, -74.0) and Tokyo (35.7, 139.7) are far

**Embeddings work the same way, but for meaning!**

- Every piece of text gets **coordinates in "meaning space"**
- Texts with **similar meanings** get **nearby coordinates**
  - "King" and "Queen" are close in meaning-space
- Texts with **different meanings** get **far-apart coordinates**
  - "King" and "Banana" are far apart in meaning-space

### 📝 What is a Vector?

This list of numbers is called a **vector** (or **embedding**). Don't let the math term scare you — a vector is just a list of numbers!

```python
# A vector is just a list of numbers!
king_embedding   = [0.2, -0.5, 0.8, 0.1, ...]  # 384 to 1536 numbers
queen_embedding  = [0.21, -0.48, 0.79, 0.12, ...]  # very similar numbers!
banana_embedding = [-0.7, 0.3, -0.2, 0.9, ...]  # very different numbers!
```

- GPS uses 2 numbers (latitude, longitude)
- Real text embeddings use **384 to 1,536 numbers** (dimensions)
- More dimensions = more nuance in capturing meaning

> 💡 **Key Insight:** An embedding model (like `text-embedding-ada-002` or `all-MiniLM-L6-v2`) is a neural network that has learned to place similar texts near each other in this high-dimensional space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Embedding Visualization
# We use simplified 2D coordinates to show how words cluster
# by meaning. Real embeddings have 384+ dimensions!
# ============================================================

# Simplified 2D embeddings (real ones have 384+ dimensions!)
words = {
    'King':     [0.9, 0.8],
    'Queen':    [0.85, 0.75],
    'Prince':   [0.8, 0.7],
    'Princess': [0.75, 0.65],
    'Apple':    [-0.5, 0.3],
    'Banana':   [-0.6, 0.25],
    'Orange':   [-0.45, 0.35],
    'Dog':      [0.1, -0.7],
    'Cat':      [0.15, -0.65],
    'Puppy':    [0.05, -0.6],
}

# Define categories for color-coding
categories = {
    'Royalty': (['King', 'Queen', 'Prince', 'Princess'], '#9C27B0', 'o'),   # purple
    'Fruits':  (['Apple', 'Banana', 'Orange'], '#4CAF50', 's'),             # green
    'Animals': (['Dog', 'Cat', 'Puppy'], '#FF9800', '^'),                   # orange
}

fig, ax = plt.subplots(figsize=(14, 8))

# Plot each category with its own color and marker
for cat_name, (word_list, color, marker) in categories.items():
    xs = [words[w][0] for w in word_list]
    ys = [words[w][1] for w in word_list]
    ax.scatter(xs, ys, c=color, marker=marker, s=200, label=cat_name,
               edgecolors='white', linewidth=1.5, zorder=5)
    # Add word labels next to each point
    for w in word_list:
        ax.annotate(w, (words[w][0], words[w][1]),
                    textcoords='offset points', xytext=(10, 8),
                    fontsize=13, fontweight='bold', color=color)

# Draw light circles around clusters to highlight grouping
from matplotlib.patches import Ellipse
ellipses = [
    {'center': (0.825, 0.725), 'w': 0.35, 'h': 0.35, 'color': '#9C27B0'},
    {'center': (-0.517, 0.3), 'w': 0.35, 'h': 0.25, 'color': '#4CAF50'},
    {'center': (0.1, -0.65), 'w': 0.25, 'h': 0.25, 'color': '#FF9800'},
]
for e in ellipses:
    ellipse = Ellipse(e['center'], e['w'], e['h'], alpha=0.12,
                      facecolor=e['color'], edgecolor=e['color'],
                      linewidth=2, linestyle='--')
    ax.add_patch(ellipse)

# Formatting
ax.set_xlabel('Dimension 1', fontsize=13)
ax.set_ylabel('Dimension 2', fontsize=13)
ax.set_title('\U0001f4cd Words as Points in "Meaning Space" (Simplified 2D)', 
             fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=12, loc='lower left', framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.0, 1.3)
ax.set_ylim(-1.0, 1.1)
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.show()

print("\n\U0001f4a1 Key observations:")
print("\u2022 Royalty words (King, Queen, Prince, Princess) cluster together \u2014 they have similar meanings")
print("\u2022 Fruit words (Apple, Banana, Orange) form their own cluster")
print("\u2022 Animal words (Dog, Cat, Puppy) are grouped together too")
print("\u2022 The distance between clusters is large \u2014 these are very different concepts")
print("\n\u2139\ufe0f  Real embeddings have 384-1536 dimensions, not just 2!")
print("   We project them down to 2D for visualization.")

---
## 📏 Vector Similarity: How to Measure Closeness

Now that we know words can be represented as points in space, we need a way to measure **how close** two points are. This tells us how similar two texts are in meaning.

### 🏠 Analogy: How Close Are Two Houses on a Map?

If you wanted to know how close two houses are, you could measure in different ways:
- **Straight-line distance** ("as the crow flies")
- **Direction they face** (do they point the same way?)
- Both have their uses!

### Three Ways to Measure Similarity

#### 1. 🎯 Cosine Similarity (Most Popular for RAG)

**What it measures:** The *angle* between two vectors (arrows from the origin).

**Intuition:** Think of two flashlights shining from the center of a room:
- If they point in the **same direction** → cosine similarity = **1.0** (identical meaning)
- If they point at **right angles** → cosine similarity = **0.0** (unrelated)
- If they point in **opposite directions** → cosine similarity = **-1.0** (opposite meaning)

**Formula:**

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Where:
- $A \cdot B$ = **dot product** (multiply corresponding numbers and add them up)
- $\|A\|$ = **magnitude of A** (the "length" of the arrow)
- $\|B\|$ = **magnitude of B**

**Why is cosine similarity so popular?**
- It only cares about *direction*, not *length*
- A long document and a short document about the same topic will have high cosine similarity
- It's fast to compute

#### 2. 📏 Euclidean Distance

**What it measures:** The straight-line distance between two points.

$$\text{distance}(A, B) = \sqrt{\sum_{i} (A_i - B_i)^2}$$

- **Smaller distance = more similar** (opposite of cosine similarity!)
- Like measuring the distance between two houses on a map

#### 3. 🔵 Dot Product

**What it measures:** Combines both direction AND magnitude.

$$A \cdot B = \sum_{i} A_i \times B_i$$

- Higher = more similar
- Used by some embedding models (e.g., OpenAI's)

> 💡 **For this course, we'll focus on cosine similarity** — it's the most widely used in RAG systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Cosine Similarity: Implementation & Visualization
# ============================================================

def cosine_similarity(a, b):
    """Calculate how similar two vectors are (like GPS coordinates for meaning).
    
    Returns a value between -1 and 1:
        1.0  = identical direction (very similar meaning)
        0.0  = perpendicular (unrelated)
       -1.0  = opposite direction (opposite meaning)
    """
    # Step 1: Dot product - multiply matching numbers and sum
    dot_product = np.dot(a, b)
    
    # Step 2: Magnitudes - the "length" of each vector
    magnitude_a = np.linalg.norm(a)
    magnitude_b = np.linalg.norm(b)
    
    # Step 3: Divide dot product by the product of magnitudes
    return dot_product / (magnitude_a * magnitude_b)


# Our simplified word embeddings from before
words = {
    'King':     np.array([0.9, 0.8]),
    'Queen':    np.array([0.85, 0.75]),
    'Prince':   np.array([0.8, 0.7]),
    'Princess': np.array([0.75, 0.65]),
    'Apple':    np.array([-0.5, 0.3]),
    'Banana':   np.array([-0.6, 0.25]),
    'Orange':   np.array([-0.45, 0.35]),
    'Dog':      np.array([0.1, -0.7]),
    'Cat':      np.array([0.15, -0.65]),
    'Puppy':    np.array([0.05, -0.6]),
}

# --- Part 1: Compute some interesting pairs ---
print("=" * 55)
print("\U0001f4cf  Cosine Similarity Between Word Pairs")
print("=" * 55)

pairs = [
    ('King', 'Queen',  'Both royalty \u2014 should be HIGH'),
    ('King', 'Prince', 'Both royalty \u2014 should be HIGH'),
    ('Dog', 'Cat',     'Both animals \u2014 should be HIGH'),
    ('Dog', 'Puppy',   'Both dogs! \u2014 should be VERY HIGH'),
    ('Apple', 'Banana', 'Both fruits \u2014 should be HIGH'),
    ('King', 'Banana', 'Royalty vs fruit \u2014 should be LOW'),
    ('Dog', 'Orange',  'Animal vs fruit \u2014 should be LOW'),
    ('King', 'Dog',    'Royalty vs animal \u2014 should be LOW'),
]

for w1, w2, explanation in pairs:
    sim = cosine_similarity(words[w1], words[w2])
    bar = '\u2588' * int(max(0, sim) * 20)  # visual bar
    print(f"  {w1:>8s} \u2194 {w2:<8s}  sim = {sim:+.4f}  {bar}  ({explanation})")

# --- Part 2: Full similarity matrix as a heatmap ---
word_names = list(words.keys())
n = len(word_names)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(words[word_names[i]], words[word_names[j]])

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')

# Add labels
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(word_names, fontsize=11, rotation=45, ha='right')
ax.set_yticklabels(word_names, fontsize=11)

# Add similarity values as text in each cell
for i in range(n):
    for j in range(n):
        val = sim_matrix[i, j]
        color = 'white' if abs(val) > 0.7 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=9, fontweight='bold', color=color)

# Colorbar and title
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Cosine Similarity', fontsize=12)
ax.set_title('\U0001f5fa\ufe0f Similarity Heatmap: How Close Are These Words in Meaning Space?',
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print("\n\U0001f4a1 Reading the heatmap:")
print("\u2022 Green cells (close to 1.0) = very similar meanings")
print("\u2022 Yellow cells (close to 0.0) = unrelated meanings")
print("\u2022 Red cells (close to -1.0) = opposite meanings")
print("\u2022 Notice the green blocks along the diagonal \u2014 every word is perfectly similar to itself (1.0)")
print("\u2022 Notice the green clusters for Royalty, Fruits, and Animals \u2014 words in the same category are similar!")

---
## 🔄 The RAG Pipeline: Step by Step

Now that we understand embeddings and similarity, let's see how they come together in a complete RAG system.

RAG has **two phases**: one that happens *once* (preparing your data), and one that happens *every time* someone asks a question.

---

### 📦 Phase 1: Indexing (Preparing Your Library) — *Happens Once*

Think of this as setting up a well-organized library before anyone walks in to ask a question.

| Step | What Happens | Analogy |
|------|-------------|--------|
| 1. **Collect Documents** | Gather all your documents (PDFs, web pages, notes, etc.) | Buying all the books for your library |
| 2. **Split into Chunks** | Break large documents into smaller, manageable pieces | Cutting a textbook into individual pages or paragraphs |
| 3. **Create Embeddings** | Convert each chunk into a vector (list of numbers) | Giving each page a GPS coordinate based on its content |
| 4. **Store in Vector DB** | Save all the vectors in a special database optimized for similarity search | Organizing your library so you can find any topic instantly |

---

### ❓ Phase 2: Querying (Answering Questions) — *Happens Every Time*

This is what happens when someone actually asks a question.

| Step | What Happens | Analogy |
|------|-------------|--------|
| 1. **User Asks Question** | "What is our refund policy?" | A student walks into the library with a question |
| 2. **Embed the Question** | Convert the question into a vector | Get the GPS coordinates of the question's *meaning* |
| 3. **Search Vector DB** | Find the stored chunks whose vectors are most similar | Find the library books closest to those GPS coordinates |
| 4. **Retrieve Top Chunks** | Pull out the top 3-5 most relevant text chunks | Pull those books off the shelf |
| 5. **Build Prompt** | Combine: question + retrieved chunks into one prompt | Hand the student the question AND the relevant books |
| 6. **LLM Generates Answer** | The LLM reads everything and writes an answer | The student reads the books and writes their answer |

> 💡 **The magic of RAG:** The LLM doesn't need to have memorized your documents. It just needs to be good at *reading and summarizing* — which it already is!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ============================================================
# Detailed RAG Pipeline Diagram (Two Phases)
# Top:    Indexing phase (prepare documents)
# Bottom: Query phase (answer questions)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ---- PHASE 1: INDEXING ----
ax1 = axes[0]
ax1.set_xlim(0, 14)
ax1.set_ylim(0, 3.5)
ax1.axis('off')
ax1.set_title('\U0001f4e6 Phase 1: Indexing (Happens Once)', fontsize=15,
              fontweight='bold', pad=10, loc='left', color='#1565C0')

index_boxes = [
    {'xy': (0.3, 0.8), 'w': 2.5, 'h': 1.6, 'color': '#42A5F5',
     'title': '\U0001f4c1 Documents', 'sub': 'PDFs, web pages,\nnotes, databases'},
    {'xy': (3.5, 0.8), 'w': 2.5, 'h': 1.6, 'color': '#66BB6A',
     'title': '\u2702\ufe0f Chunking', 'sub': 'Split into smaller\npieces (paragraphs)'},
    {'xy': (6.7, 0.8), 'w': 2.5, 'h': 1.6, 'color': '#FFA726',
     'title': '\U0001f522 Embedding', 'sub': 'Convert chunks\nto vectors'},
    {'xy': (9.9, 0.8), 'w': 3.5, 'h': 1.6, 'color': '#AB47BC',
     'title': '\U0001f4be Vector Database', 'sub': 'Store vectors for\nfast similarity search'},
]

for box in index_boxes:
    rect = mpatches.FancyBboxPatch(
        box['xy'], box['w'], box['h'],
        boxstyle='round,pad=0.1', facecolor=box['color'],
        edgecolor='white', linewidth=2, alpha=0.85
    )
    ax1.add_patch(rect)
    cx = box['xy'][0] + box['w'] / 2
    cy = box['xy'][1] + box['h'] / 2
    ax1.text(cx, cy + 0.35, box['title'], ha='center', va='center',
             fontsize=11, fontweight='bold', color='white')
    ax1.text(cx, cy - 0.25, box['sub'], ha='center', va='center',
             fontsize=9, color='white', style='italic')

# Arrows for indexing phase
arrow_style_idx = dict(arrowstyle='->', lw=2.5, color='#333333')
idx_arrows = [(2.8, 3.5), (6.0, 6.7), (9.2, 9.9)]
for (xs, xe) in idx_arrows:
    ax1.annotate('', xy=(xe, 1.6), xytext=(xs, 1.6), arrowprops=arrow_style_idx)

# ---- PHASE 2: QUERYING ----
ax2 = axes[1]
ax2.set_xlim(0, 14)
ax2.set_ylim(0, 3.5)
ax2.axis('off')
ax2.set_title('\u2753 Phase 2: Querying (Happens Every Time)', fontsize=15,
              fontweight='bold', pad=10, loc='left', color='#C62828')

query_boxes = [
    {'xy': (0.1, 0.8), 'w': 1.8, 'h': 1.6, 'color': '#EF5350',
     'title': '\U0001f64b Question', 'sub': 'User asks\na question'},
    {'xy': (2.4, 0.8), 'w': 1.8, 'h': 1.6, 'color': '#FFA726',
     'title': '\U0001f522 Embed', 'sub': 'Question\n\u2192 vector'},
    {'xy': (4.7, 0.8), 'w': 1.8, 'h': 1.6, 'color': '#AB47BC',
     'title': '\U0001f50d Search', 'sub': 'Find similar\nvectors in DB'},
    {'xy': (7.0, 0.8), 'w': 1.8, 'h': 1.6, 'color': '#66BB6A',
     'title': '\U0001f4c4 Retrieve', 'sub': 'Get matching\ntext chunks'},
    {'xy': (9.3, 0.8), 'w': 1.8, 'h': 1.6, 'color': '#42A5F5',
     'title': '\U0001f9e0 LLM', 'sub': 'Question +\nchunks \u2192 LLM'},
    {'xy': (11.6, 0.8), 'w': 2.0, 'h': 1.6, 'color': '#26A69A',
     'title': '\u2705 Answer', 'sub': 'Grounded\nresponse'},
]

for box in query_boxes:
    rect = mpatches.FancyBboxPatch(
        box['xy'], box['w'], box['h'],
        boxstyle='round,pad=0.1', facecolor=box['color'],
        edgecolor='white', linewidth=2, alpha=0.85
    )
    ax2.add_patch(rect)
    cx = box['xy'][0] + box['w'] / 2
    cy = box['xy'][1] + box['h'] / 2
    ax2.text(cx, cy + 0.35, box['title'], ha='center', va='center',
             fontsize=11, fontweight='bold', color='white')
    ax2.text(cx, cy - 0.25, box['sub'], ha='center', va='center',
             fontsize=9, color='white', style='italic')

# Arrows for query phase
q_arrows = [(1.9, 2.4), (4.2, 4.7), (6.5, 7.0), (8.8, 9.3), (11.1, 11.6)]
for (xs, xe) in q_arrows:
    ax2.annotate('', xy=(xe, 1.6), xytext=(xs, 1.6), arrowprops=arrow_style_idx)

plt.tight_layout()
plt.show()

print("\n\U0001f4a1 The two phases of RAG:")
print("\n\U0001f4e6 INDEXING (top) \u2014 Prepare your documents once:")
print("   Documents \u2192 Chunk them \u2192 Embed each chunk \u2192 Store in vector DB")
print("\n\u2753 QUERYING (bottom) \u2014 Answer questions on demand:")
print("   Question \u2192 Embed it \u2192 Search DB \u2192 Retrieve chunks \u2192 LLM generates answer")

---
## ⚖️ RAG vs Fine-Tuning

A common question is: *"Why not just fine-tune the model on my data instead?"*

Great question! Here's a comparison:

| Feature | RAG | Fine-Tuning |
|---------|-----|-------------|
| 💰 **Cost** | Lower (no training needed) | Higher (requires GPU training) |
| 🔄 **Updating Knowledge** | Easy — just add new documents | Hard — must retrain the model |
| 🔍 **Transparency** | High — can show source documents | Low — knowledge is baked into weights |
| 🎯 **Best For** | Answering questions about specific documents | Changing model behavior/style |
| ⏱️ **Setup Time** | Hours | Days to weeks |
| 📊 **Data Needed** | Any amount of text | Hundreds to thousands of examples |
| 🧠 **How It Works** | Gives the model information at query time | Changes the model's internal weights |

### 🏠 Analogy

- **RAG** = Giving a smart student an open book during the test
- **Fine-tuning** = Sending a student to a special training school before the test

Both are valuable! And you can even combine them. But RAG is usually the **first thing to try** because it's faster, cheaper, and easier to update.

> 💡 **Rule of thumb:** Start with RAG. Only fine-tune if RAG isn't enough (e.g., you need the model to adopt a specific writing style or learn a specialized reasoning pattern).

In [ ]:
import numpy as np

# ============================================================
# Mini RAG System from Scratch (No External Dependencies!)
# 
# We'll build a tiny but complete RAG pipeline using only numpy.
# This demonstrates the core concepts without needing any
# external embedding models or vector databases.
# ============================================================

print("\U0001f680 Building a Mini RAG System from Scratch!")
print("=" * 55)

# --- Step 1: Our "document database" (knowledge base) ---
# In real RAG, these would be chunks from your actual documents
documents = [
    "The cheetah is the fastest land animal, reaching speeds up to 70 mph.",
    "Blue whales are the largest animals ever to have lived on Earth, reaching lengths of up to 100 feet.",
    "Octopuses have three hearts and blue blood. Two hearts pump blood to the gills, one to the rest of the body.",
    "Honey never spoils. Archaeologists have found 3000-year-old honey in Egyptian tombs that was still edible.",
    "Dogs can understand up to 250 words and gestures, and can count up to five.",
    "The Amazon rainforest produces about 20 percent of the world's oxygen supply.",
]

print("\n\U0001f4da Our Knowledge Base (6 document chunks):")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc}")


# --- Step 2: Create simple embeddings ---
# Real systems use neural network embedding models (like sentence-transformers).
# Here we'll use a simplified TF-IDF-like approach: count important words!

def simple_embedding(text, vocabulary):
    """Create a simple embedding by counting how often each vocabulary word appears.
    
    This is a toy version of what real embedding models do.
    Real models use neural networks to capture deep semantic meaning.
    """
    text_lower = text.lower()
    # Count occurrences of each vocabulary word
    vector = np.array([text_lower.count(word) for word in vocabulary], dtype=float)
    # Normalize to unit length (like real embedding models do)
    norm = np.linalg.norm(vector)
    if norm > 0:
        vector = vector / norm
    return vector


def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


# Build a simple vocabulary from key words across our documents
vocabulary = [
    'fast', 'speed', 'animal', 'land', 'cheetah',
    'large', 'biggest', 'whale', 'blue', 'ocean', 'sea',
    'heart', 'blood', 'octopus',
    'honey', 'food', 'old', 'ancient', 'spoil',
    'dog', 'smart', 'word', 'understand', 'count',
    'forest', 'oxygen', 'tree', 'amazon', 'rainforest',
]

print(f"\n\U0001f4dd Vocabulary size: {len(vocabulary)} words")
print(f"   (Real embedding models capture meaning in 384-1536 dimensions)\n")


# --- Step 3: Embed all documents (this is the "indexing" phase) ---
print("\U0001f4e6 Phase 1: INDEXING \u2014 Embedding all documents...")
doc_embeddings = []
for doc in documents:
    emb = simple_embedding(doc, vocabulary)
    doc_embeddings.append(emb)

print(f"   \u2713 Embedded {len(documents)} documents into {len(vocabulary)}-dimensional vectors")
print(f"   \u2713 Example vector for doc[0]: [{', '.join(f'{x:.2f}' for x in doc_embeddings[0][:8])}...]")


# --- Step 4: Query time! (this is the "querying" phase) ---
def search(query, documents, doc_embeddings, vocabulary, top_k=3):
    """Search for the most relevant documents given a query.
    
    This is the core retrieval step in RAG:
    1. Embed the query
    2. Compare to all document embeddings
    3. Return the most similar ones
    """
    # Embed the query using the same method
    query_emb = simple_embedding(query, vocabulary)
    
    # Calculate similarity to every document
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_emb, doc_emb)
        similarities.append((i, sim))
    
    # Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return top-k results
    return similarities[:top_k]


# --- Step 5: Try some queries! ---
queries = [
    "What is the fastest animal?",
    "Tell me about octopus hearts and blood",
    "Which animal is the smartest and can understand words?",
]

print("\n" + "=" * 55)
print("\u2753 Phase 2: QUERYING \u2014 Let's search!")
print("=" * 55)

for query in queries:
    print(f"\n\U0001f50e Query: \"{query}\"")
    results = search(query, documents, doc_embeddings, vocabulary, top_k=3)
    
    print(f"   Top {len(results)} results:")
    for rank, (doc_idx, similarity) in enumerate(results, 1):
        bar = '\u2588' * int(max(0, similarity) * 30)
        print(f"   {rank}. [sim={similarity:.3f}] {bar}")
        print(f"      \"{documents[doc_idx]}\"")
    
    # Show what would happen in a real RAG system
    top_doc_idx = results[0][0]
    print(f"\n   \u27a1\ufe0f In a real RAG system, the retrieved chunk(s) would be sent to an LLM:")
    print(f"   \u250c\u2500 Prompt to LLM \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(f"   \u2502 Context: {documents[top_doc_idx]}")
    print(f"   \u2502 Question: {query}")
    print(f"   \u2502 Please answer based on the context above.")
    print(f"   \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")

---
## ✅ When to Use RAG

RAG is the right choice when:

- ✅ **Your data changes frequently** — news articles, updated docs, evolving knowledge base
- ✅ **You need answers grounded in specific documents** — company policies, research papers, legal contracts
- ✅ **You want to cite sources** — show users *where* the answer came from
- ✅ **You have private/proprietary data** — internal wikis, customer records, private repos
- ✅ **You want to reduce hallucinations** — giving the LLM factual context helps a lot

RAG is **NOT** the best choice when:

- ❌ **You need the model to learn a new skill or style** — use fine-tuning instead
- ❌ **Your data is very small** (a few sentences) — just paste it directly in the prompt
- ❌ **You need real-time data** (stock prices, live feeds) — use API calls / tool use instead
- ❌ **The task is purely creative** (writing fiction) — no reference docs needed

---
## ⚠️ Common Misconceptions

Let's bust some myths about RAG:

### ❌ "RAG makes LLMs smarter"
**Reality:** RAG doesn't change the LLM at all. It gives the LLM better *reference material*. The LLM's intelligence stays the same — it just has better information to work with. Think of it like giving a student better textbooks. The student doesn't get smarter; they just have better resources.

### ❌ "RAG eliminates hallucinations"
**Reality:** RAG *significantly reduces* hallucinations by grounding answers in retrieved documents, but it doesn't eliminate them entirely. The LLM can still:
- Misinterpret the retrieved text
- Combine information incorrectly
- Fill in gaps with guesses if the retrieved context doesn't fully answer the question

### ❌ "Any chunking strategy works"
**Reality:** How you split your documents into chunks has a *huge* impact on quality. Chunks that are too small lose context. Chunks that are too big dilute the relevant information. Finding the right chunking strategy is one of the most important parts of building a good RAG system. (We'll cover this in the next notebook!)

### ❌ "More retrieved documents = better answers"
**Reality:** Retrieving too many chunks can actually *hurt* performance:
- The LLM has a limited context window
- Irrelevant chunks add noise and can confuse the model
- The model may struggle to find the key information in a sea of text
- Typically, **3-5 well-chosen chunks** outperform 20 mediocre ones

---
## 🎯 Key Takeaways

Let's summarize everything we've learned:

1. **The Problem:** LLMs have knowledge cutoffs, can hallucinate, and don't know about your specific data. They're like students taking a closed-book test.

2. **The Solution — RAG:** Retrieval-Augmented Generation gives the LLM an "open book" by retrieving relevant documents at query time.
   - **R**etrieval: Find relevant information
   - **A**ugmented: Add it to the prompt
   - **G**eneration: LLM generates a grounded answer

3. **Embeddings:** Text is converted into lists of numbers (vectors) that capture meaning. Similar texts have similar vectors.

4. **Vector Similarity:** We use cosine similarity to measure how close two vectors are. Values range from -1 (opposite) to 1 (identical).

5. **The RAG Pipeline has two phases:**
   - **Indexing** (once): Documents → Chunks → Embeddings → Vector Database
   - **Querying** (every time): Question → Embed → Search → Retrieve → LLM → Answer

6. **RAG vs Fine-tuning:** RAG is cheaper, easier to update, and more transparent. Fine-tuning is better for changing model behavior. Start with RAG.

7. **RAG isn't magic:** It reduces hallucinations but doesn't eliminate them. Chunking strategy and retrieval quality matter a lot.

---
## 🤔 Test Your Understanding

Before moving on, test yourself with these questions:

**1. What does RAG stand for, and what does each word mean?**
<details>
<summary>Click to reveal answer</summary>

**R**etrieval-**A**ugmented **G**eneration.
- **Retrieval:** Finding relevant documents from a knowledge base
- **Augmented:** Enhancing the LLM's prompt with the retrieved information
- **Generation:** The LLM generating an answer based on the question plus the retrieved context
</details>

**2. What is an embedding, and why do we need it?**
<details>
<summary>Click to reveal answer</summary>

An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. We need embeddings because computers can't understand text directly — they need numbers. Embeddings let us calculate how similar two pieces of text are by comparing their vectors.
</details>

**3. If the cosine similarity between "dog" and "puppy" is 0.95, and between "dog" and "refrigerator" is 0.05, what does that tell us?**
<details>
<summary>Click to reveal answer</summary>

It tells us that "dog" and "puppy" have very similar meanings (0.95 is close to 1.0), which makes sense because a puppy IS a young dog. Meanwhile, "dog" and "refrigerator" are almost completely unrelated in meaning (0.05 is close to 0.0). The embedding model has learned that dogs and puppies are semantically related, while dogs and refrigerators are not.
</details>

**4. What are the two phases of a RAG pipeline, and when does each one run?**
<details>
<summary>Click to reveal answer</summary>

1. **Indexing Phase** (runs once, or when documents change): Documents are collected, split into chunks, converted to embeddings, and stored in a vector database.
2. **Query Phase** (runs every time a user asks a question): The question is embedded, the vector database is searched for similar chunks, the most relevant chunks are retrieved, and everything is sent to the LLM to generate an answer.
</details>

**5. Why might RAG be a better starting point than fine-tuning for a document Q&A system?**
<details>
<summary>Click to reveal answer</summary>

RAG is typically better as a starting point because:
- It's **cheaper** (no training required)
- It's **easier to update** (just add new documents, no retraining)
- It's **more transparent** (you can show users which source documents were used)
- It requires **less data** to get started
- It's **faster to set up** (hours instead of days)

Fine-tuning is better when you need to change the model's *behavior* or *style*, not just give it more information.
</details>

---
## 🚀 What's Next?

Congratulations! You now understand what RAG is, how embeddings work, and the complete pipeline from start to finish. 🎉

In the next notebook, we'll dive into one of the most critical aspects of building a great RAG system: **how to split your documents into chunks**. The quality of your chunks directly determines the quality of your retrieval, which determines the quality of your answers.

You'll learn:
- Different chunking strategies (fixed-size, sentence-based, semantic)
- How chunk size affects retrieval quality
- Overlap strategies to avoid losing context at chunk boundaries
- Practical tips for choosing the right approach

**Ready to continue?** → [Continue to Notebook 2: Chunking Techniques](02_chunking_techniques.ipynb)

---

### 📚 Optional Further Reading

Want to explore more before moving on?

- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) — The original RAG paper by Lewis et al. (2020)
- [What is RAG? (IBM)](https://research.ibm.com/blog/retrieval-augmented-generation-RAG) — Great visual overview
- [Sentence Transformers](https://www.sbert.net/) — Popular library for creating text embeddings
- [FAISS: A Library for Efficient Similarity Search](https://github.com/facebookresearch/faiss) — Facebook's vector search library

---

*Great job on completing Notebook 1! You're on your way to mastering RAG! 🌟*